# Distribution reliability indices from an outage log

This notebook computes the IEEE Std 1366 distribution reliability indices (SAIFI, SAIDI, CAIDI, MAIFI_E, MAIFI, and ASAI) from a committed synthetic interruption record.

These are **distribution-reliability indices**, not resilience metrics. They measure how often and how long service was interrupted under past conditions. The EPRI Climate READi forward-looking metrics (CORi, LORi, DORi) need fragility curves and hazard probabilities that `climate_indices` does not model.

The input here is the committed fixture `tests/fixture/reliability/events.json`; the notebook needs no download, credentials, or network access. For the equations and the conventions that vary between utilities, see `docs/algorithm_refs/reliability.md`.


In [1]:
from pathlib import Path
import json
import math

import pandas as pd

from climate_indices import reliability

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

events = json.loads((ROOT / "tests/fixture/reliability/events.json").read_text())
pd.DataFrame(events)[["event_id", "start_time", "duration", "customers_interrupted", "device_operations"]]


,event_id,start_time,duration,customers_interrupted,device_operations
0,sustained-1,2024-01-01T00:00:00Z,60.0,100,1
1,sustained-2,2024-01-01T02:00:00Z,90.0,200,1
2,momentary-1,2024-01-01T04:00:00Z,1.0,150,2
3,momentary-2,2024-01-01T05:00:00Z,2.0,150,3
4,zero-duration,2024-01-01T06:00:00Z,0.0,0,0


## Classify the record

Every row is sustained or momentary by its duration. The default threshold is five minutes; a duration strictly above it is sustained. `classify_interruptions()` returns a normalized copy without computing metrics.


In [2]:
classified = reliability.classify_interruptions(events)
classified[["event_id", "duration", "customers_interrupted", "interruption_type"]]


,event_id,duration,customers_interrupted,interruption_type
0,sustained-1,60.0,100,sustained
1,sustained-2,90.0,200,sustained
2,momentary-1,1.0,150,momentary
3,momentary-2,2.0,150,momentary
4,zero-duration,0.0,0,momentary


## Compute the metrics

`reliability_indices()` validates the record, classifies each row, and computes every metric once from two shared accumulators: `CI = sum(N_i)` and `CMI = sum(r_i * N_i)` over sustained rows. The served-customer count is 1000 and the reporting period is 24 hours.


In [3]:
result = reliability.reliability_indices(events, total_customers=1000, period_hours=24)
result


ReliabilityResult(saifi=0.3, customer_interruptions=300, outage_count=2, saidi=24.0, customer_minutes_interrupted=24000.0, maifi_e=0.3, maifi=0.75, period_hours=24.0, momentary_outage_count=3)

The two sustained records give 300 customer interruptions and 24,000 customer-minutes. SAIFI is 0.3 interruptions per customer, SAIDI is 24.0 minutes per customer, and CAIDI is their ratio, 80.0 minutes per interruption. ASAI is the available fraction of customer-hours, `1 - (24 / 60) / 24`, about 0.9833.

The two momentary records give 300 momentary events and 750 device operations, so MAIFI_E is 0.3 and MAIFI is 0.75. The final zero-duration row is momentary and raises `momentary_outage_count` to three, but it contributes nothing to the sustained metrics.


In [4]:
summary = pd.DataFrame(
    {
        "metric": ["saifi", "saidi", "caidi", "asai", "maifi_e", "maifi"],
        "value": [
            result.saifi,
            result.saidi,
            result.caidi,
            result.asai,
            result.maifi_e,
            result.maifi,
        ],
    }
)
summary["units"] = [
    "interruptions per customer",
    "minutes per customer",
    "minutes per interruption",
    "fraction available",
    "momentary events per customer",
    "device operations per customer",
]
summary


,metric,value,units
0,saifi,0.300000,interruptions per customer
1,saidi,24.000000,minutes per customer
2,caidi,80.000000,minutes per interruption
3,asai,0.983333,fraction available
4,maifi_e,0.300000,momentary events per customer
5,maifi,0.750000,device operations per customer


## Undefined metrics and the reporting period

ASAI needs an explicit reporting period. Without one it is `None`, and `undefined_metrics` names it. The library never assumes 8760 hours and never infers the period from event timestamps. Period bounds give the same actual elapsed hours, including leap days and daylight-saving changes.


In [5]:
without_period = reliability.reliability_indices(events, total_customers=1000)
print("asai without a period:", without_period.asai)
print("undefined:", without_period.undefined_metrics)

from_bounds = reliability.reliability_indices(
    events,
    total_customers=1000,
    period_start="2024-01-01T00:00:00Z",
    period_end="2024-01-02T00:00:00Z",
)
print("asai from bounds:", from_bounds.asai)
print("period hours:", from_bounds.period_hours)


asai without a period: None
undefined: ('asai',)
asai from bounds: 0.9833333333333333
period hours: 24.0


Partial periods are computed exactly over the supplied period, with **no annualization and no extrapolation**. A short-period result is not comparable with a full-year result, or with a partial-period result of a different length.


In [6]:
year = reliability.reliability_indices(
    [],
    total_customers=1000,
    period_start="2024-01-01T00:00:00Z",
    period_end="2025-01-01T00:00:00Z",
)
day = reliability.reliability_indices(
    [],
    total_customers=1000,
    period_start="2024-01-01T00:00:00Z",
    period_end="2024-01-02T00:00:00Z",
)
print("leap year hours:", year.period_hours)
print("one day hours:", day.period_hours)


leap year hours: 8784.0
one day hours: 24.0


## Group a raw operation log into momentary events

`reliability_indices()` never regroups a caller's rows. `group_momentary_events()` turns a raw reclosing-operation log into one row per event per device and grouping window. The grouping window is separate from the classification threshold.


In [7]:
operations = [
    {"event_id": "op-1", "start_time": "2024-01-01T04:00:00Z", "duration": 0.0,
     "customers_interrupted": 150, "device_id": "recloser-a", "device_operations": 1},
    {"event_id": "op-2", "start_time": "2024-01-01T04:03:00Z", "duration": 0.0,
     "customers_interrupted": 150, "device_id": "recloser-a", "device_operations": 1},
]
grouped = reliability.group_momentary_events(operations, threshold_minutes=1.0)
grouped[["event_id", "duration", "customers_interrupted", "device_operations", "grouped_momentary_event"]]


,event_id,duration,customers_interrupted,device_operations,grouped_momentary_event
0,op-1,3.0,150,2,True


Two operations three minutes apart on the same device collapse into one momentary event with an elapsed duration of 3.0 minutes and two device operations.

For the full field contract and every edge case, see the reliability section of the API reference (`docs/reference.md`) and `docs/algorithm_refs/reliability.md`, and the evidence tier in `VALIDATION.md`.
